# Fuse-Traffic: Transformer-Based Event Verification Training

This notebook trains the tiered verification system for Metro Manila: 
1) Translate Tagalog/Taglish -> English (MarianMT)
2) Pre-clean text
3) DistilBERT (Event vs Non-Event)
4) RoBERTa (Reliability/Severity)
5) Tiered decision logic + metrics

**Note:** You do not have labels yet, so this notebook includes a labeling stub and re-load step.

## 1. Dependencies & Setup

In [ ]:
!pip -q install transformers torch sentencepiece sacremoses evaluate accelerate

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Data Loading & Labeling Stub

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
from pathlib import Path

# Update this path if you copied the file to Drive
X_CSV_PATH = Path('/content/drive/MyDrive/x_recent_posts_translated.csv')

df = pd.read_csv(X_CSV_PATH)
df.head()

### Labeling Stub (Manual)
Export a CSV for manual labeling.

- `event_label`: 1 for traffic/event, 0 for non-event
- `reliability_label`: 0 discard, 1 low reliability, 2 high reliability

In [ ]:
LABEL_STUB_PATH = Path('/content/drive/MyDrive/x_label_stub.csv')

label_stub = df[[
]].copy()
label_stub['event_label'] = ''
label_stub['reliability_label'] = ''
label_stub.to_csv(LABEL_STUB_PATH, index=False)

print('Saved stub:', LABEL_STUB_PATH)

After labeling, re-load the file below.

In [ ]:
LABELED_PATH = Path('/content/drive/MyDrive/x_label_stub.csv')
labeled_df = pd.read_csv(LABELED_PATH)
labeled_df.head()

## 3. Text Preprocessing & Translation

In [ ]:
import re
from transformers import MarianMTModel, MarianTokenizer

class TextPreprocessor:
    def __init__(self, model_name='Helsinki-NLP/opus-mt-tl-en', device=None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.tokenizer = MarianTokenizer.from_pretrained(model_name)
        self.model = MarianMTModel.from_pretrained(model_name).to(self.device)

    def clean(self, text):
        text = re.sub(r'http\S+|www\.\S+', '', str(text))
        text = re.sub(r'@\w+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def translate_batch(self, texts, batch_size=16):
        outputs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            inputs = self.tokenizer(batch, return_tensors='pt', padding=True, truncation=True).to(self.device)
            with torch.no_grad():
                generated = self.model.generate(**inputs, max_length=256)
            outputs.extend(self.tokenizer.batch_decode(generated, skip_special_tokens=True))
        return outputs

pre = TextPreprocessor()

labeled_df['clean_text'] = labeled_df['raw_text'].apply(pre.clean)
labeled_df['translated_text'] = pre.translate_batch(labeled_df['clean_text'].tolist())

labeled_df.head()

## 4. DistilBERT Fine-Tuning (Binary Classification)

In [ ]:
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
import evaluate

binary_df = labeled_df.dropna(subset=['event_label']).copy()
binary_df['event_label'] = binary_df['event_label'].astype(int)

dataset = Dataset.from_pandas(binary_df[['translated_text', 'event_label']])
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def tokenize_batch(batch):
    return tokenizer(batch['translated_text'], truncation=True, padding=True)

dataset = dataset.map(tokenize_batch, batched=True)
dataset = dataset.rename_column('event_label', 'labels')
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

split = dataset.train_test_split(test_size=0.2, seed=42)
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=preds, references=labels)['accuracy'],
        'f1': f1.compute(predictions=preds, references=labels)['f1']
    }

model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

args = TrainingArguments(
    output_dir='/content/distilbert_event',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

## 5. RoBERTa Fine-Tuning (Reliability Scoring)

In [ ]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification
from torch.nn import CrossEntropyLoss
import numpy as np

multi_df = labeled_df.dropna(subset=['reliability_label']).copy()
multi_df['reliability_label'] = multi_df['reliability_label'].astype(int)

dataset_m = Dataset.from_pandas(multi_df[['translated_text', 'reliability_label']])
tokenizer_m = RobertaTokenizerFast.from_pretrained('roberta-base')

def tokenize_batch_m(batch):
    return tokenizer_m(batch['translated_text'], truncation=True, padding=True)

dataset_m = dataset_m.map(tokenize_batch_m, batched=True)
dataset_m = dataset_m.rename_column('reliability_label', 'labels')
dataset_m.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

split_m = dataset_m.train_test_split(test_size=0.2, seed=42)

class_weights = np.bincount(multi_df['reliability_label'])
class_weights = class_weights.sum() / (len(class_weights) * class_weights)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(model.device)

model_m = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=3)

def compute_loss(model, inputs, return_outputs=False):
    labels = inputs.get('labels')
    outputs = model(**inputs)
    logits = outputs.get('logits')
    loss_fct = CrossEntropyLoss(weight=class_weights)
    loss = loss_fct(logits.view(-1, 3), labels.view(-1))
    return (loss, outputs) if return_outputs else loss

args_m = TrainingArguments(
    output_dir='/content/roberta_reliability',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
)

trainer_m = Trainer(
    model=model_m,
    args=args_m,
    train_dataset=split_m['train'],
    eval_dataset=split_m['test'],
    tokenizer=tokenizer_m,
    compute_loss=compute_loss,
)

trainer_m.train()
trainer_m.evaluate()

## 6. Tiered Verification Logic

In [ ]:
import numpy as np

DISTIL_THRESHOLD = 0.85
ROBERTA_THRESHOLD = 0.6

def tiered_decision(distil_score, roberta_score):
    if distil_score >= DISTIL_THRESHOLD:
        return 1
    if roberta_score >= ROBERTA_THRESHOLD:
        return 1
    return 0

# Placeholder arrays for combined evaluation
distil_scores = np.array([0.9, 0.2, 0.7])
roberta_scores = np.array([0.1, 0.8, 0.3])
combined = [tiered_decision(d, r) for d, r in zip(distil_scores, roberta_scores)]
combined

## 7. Metrics & Export

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

# Replace with real predictions and labels
y_true = np.array([1, 0, 1])
y_pred = np.array(combined)

precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
auc = roc_auc_score(y_true, y_pred)

print('Precision:', precision)
print('Recall:', recall)
print('F1:', f1)
print('AUC:', auc)

model.save_pretrained('/content/distilbert_event')
tokenizer.save_pretrained('/content/distilbert_event')
model_m.save_pretrained('/content/roberta_reliability')
tokenizer_m.save_pretrained('/content/roberta_reliability')

print('Models saved.')